# Model Combination Sweep — Which Subset of Models Gives the Best Ensemble?

**Objective**: The full 7-model ensemble may not be optimal. Some models may add noise
or drag down the ensemble. This notebook tests every possible combination of 2, 3, 4, 5,
6, and 7 models across all ensemble strategies to find the best subset.

**Grid**:
- Model subsets: C(7,2)=21 + C(7,3)=35 + C(7,4)=35 + C(7,5)=21 + C(7,6)=7 + C(7,7)=1 = **120 combinations**
- Ensemble strategies: 6 (Simple Mean, Trimmed Mean, Median, Geometric Mean, Inv-MAE Weighted, Rank Fusion)
- Total experiments: **720**

**Approach**: Run LOOCV once for all 7 models, then recombine predictions in-memory.
This is computationally efficient — no model retraining needed for each combination.

---
## Section 1 — Setup and Data Loading

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'ensemble_analysis':
    os.chdir('..')
elif not os.path.exists('data/processed'):
    for cand in ['.', '..', 'mari_poc', '../mari_poc']:
        if os.path.exists(os.path.join(cand, 'data/processed')):
            os.chdir(cand); break
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from itertools import combinations
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from lifelines import CoxPHFitter, WeibullAFTFitter
import xgboost as xgb
from sksurv.ensemble import (GradientBoostingSurvivalAnalysis,
                              ComponentwiseGradientBoostingSurvivalAnalysis)
from sksurv.util import Surv
from scipy.stats import gmean

DATA_DIR = Path('data/processed')
FIG_DIR  = Path('ensemble_analysis/figures')
RES_DIR  = Path('ensemble_analysis/results')
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

GWC_RKB_M = 754.0; CAP = 1200
WGR_THRESHOLD = 5.0; BT_SUSTAINED = 3

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
})
print(f'Working directory: {os.getcwd()}')

Working directory: /Users/saifullahkatpar/Downloads/Projects/Water-Breakthrough/water-breakthrough-prediction/mari_poc


---
## Section 2 — Data and Feature Engineering

In [2]:
panel  = pd.read_csv(DATA_DIR / 'panel_long.csv', parse_dates=['date'])
static = pd.read_csv(DATA_DIR / 'well_static.csv',
                      parse_dates=['first_prod_date', 'last_prod_date'])

EXCLUDED = {'M-51-HRL'}
vert_wells = static.loc[~static['is_horizontal'] &
                         ~static['well'].isin(EXCLUDED), 'well'].tolist()
panel_v  = panel[panel['well'].isin(vert_wells)].copy()
static_v = static[static['well'].isin(vert_wells)].copy().reset_index(drop=True)

def detect_breakthrough(pdf, threshold=WGR_THRESHOLD, sustained=BT_SUSTAINED):
    rows = []
    for well, grp in pdf.groupby('well'):
        grp = grp.sort_values('date').reset_index(drop=True)
        gas = grp['gas_mmcf'].values
        water = grp['water_bbl'].fillna(0).values
        wgr = np.where(gas > 0, water / gas, np.nan)
        above = np.where(np.isnan(wgr), 0, (wgr > threshold).astype(int))
        bt_found, bt_idx, rs, rl = False, None, None, 0
        for k, v in enumerate(above):
            if v:
                if rs is None: rs = k
                rl += 1
                if rl >= sustained: bt_found, bt_idx = True, rs; break
            else: rs, rl = None, 0
        fg = grp.loc[grp['gas_mmcf'] > 0]
        fp = fg['date'].iloc[0] if len(fg) else grp['date'].iloc[0]
        if bt_found:
            bd = grp.loc[bt_idx, 'date']
            tte = (bd.year * 12 + bd.month) - (fp.year * 12 + fp.month)
        else:
            ld = grp['date'].iloc[-1]
            tte = (ld.year * 12 + ld.month) - (fp.year * 12 + fp.month)
            bd = pd.NaT
        rows.append({'well': well, 'spud_date': fp, 'bt_date': bd,
                     'tte_months': max(tte, 1), 'event': int(bt_found)})
    return pd.DataFrame(rows)

def _mi(dates, ref):
    return (dates.dt.year * 12 + dates.dt.month) - (ref.year * 12 + ref.month)

def build_features(coh, pan):
    df = coh.copy()
    df['log_perm'] = np.log10(df['permeability_md'])
    df['perf_midpoint_md'] = (df['top_perf_md'] + df['bottom_perf_md']) / 2
    df['perf_thickness_md'] = df['bottom_perf_md'] - df['top_perf_md']
    df['dist_to_gwc_m'] = GWC_RKB_M - df['perf_midpoint_md']
    df['spud_year'] = (df['first_prod_date'].dt.year +
                       df['first_prod_date'].dt.month / 12)
    for c in ['early_gas_rate', 'peak_gas_rate', 'cum_gas_year1',
              'cum_gas_year3', 'initial_whfp', 'whfp_decline_rate',
              'gas_rate_cv_yr12']:
        df[c] = np.nan
    all_w = df['well'].tolist()
    for idx, row in df.iterrows():
        wp = pan[pan['well'] == row['well']].sort_values('date')
        fg = wp.loc[wp['gas_mmcf'] > 0]
        if len(fg) == 0: continue
        fp = fg['date'].iloc[0]; mi = _mi(wp['date'], fp)
        e6 = wp[(mi >= 0) & (mi < 6) & (wp['gas_mmcf'] > 0)]
        if len(e6):
            df.at[idx, 'early_gas_rate'] = (e6['gas_mmcf'] /
                                             e6['prod_days'].fillna(30)).mean()
        df.at[idx, 'peak_gas_rate'] = wp['gas_mmcf'].max()
        g24 = wp.loc[(mi >= 0) & (mi < 24), 'gas_mmcf']
        if len(g24) >= 6 and g24.mean() > 0:
            df.at[idx, 'gas_rate_cv_yr12'] = g24.std() / g24.mean()
    df['field_cum_gas_at_spud'] = np.nan
    df['n_wells_producing_at_spud'] = np.nan
    for idx, row in df.iterrows():
        spud = pd.Timestamp(row['spud_date'])
        oth = pan[(pan['well'] != row['well']) & pan['well'].isin(all_w) &
                  (pan['date'] < spud)]
        df.at[idx, 'field_cum_gas_at_spud'] = oth['gas_mmcf'].sum() / 1000
        near = pan[(pan['well'] != row['well']) & pan['well'].isin(all_w) &
                   (pan['date'] >= spud - pd.DateOffset(months=1)) &
                   (pan['date'] <= spud) & (pan['gas_mmcf'] > 0)]
        df.at[idx, 'n_wells_producing_at_spud'] = near['well'].nunique()
    return df

bt_df  = detect_breakthrough(panel_v)
cohort = static_v.merge(bt_df, on='well')
cohort = build_features(cohort, panel_v)
FEATURES = ['n_wells_producing_at_spud', 'spud_year', 'field_cum_gas_at_spud',
            'gas_rate_cv_yr12', 'peak_gas_rate']

print(f"Cohort: {len(cohort)} wells, {int(cohort['event'].sum())} events")

Cohort: 16 wells, 12 events


---
## Section 3 — LOOCV: All 7 Individual Models

Run LOOCV once. Store per-well predictions for all models.
All subsequent combination experiments reuse these predictions.

In [3]:
MODEL_NAMES = ['Cox PH', 'Weibull AFT', 'XGB Cox', 'XGB AFT',
               'sksurv GBSA', 'sksurv CWGB', 'Stacked']
# Short labels for display
MODEL_SHORT = {'Cox PH': 'Cox', 'Weibull AFT': 'Weib', 'XGB Cox': 'XGBc',
               'XGB AFT': 'XGBa', 'sksurv GBSA': 'GBSA', 'sksurv CWGB': 'CWGB',
               'Stacked': 'Stk'}

def _safe_pm(model, X, cap=CAP):
    raw = model.predict_median(X)
    arr = np.atleast_1d(np.array(raw, dtype=float))
    return np.where(np.isinf(arr) | np.isnan(arr) | (arr > cap), cap, arr)

def harrell_cindex(actual, predicted, events):
    actual = np.asarray(actual, float)
    predicted = np.asarray(predicted, float)
    events = np.asarray(events, int)
    con = dis = tie = 0
    for i in range(len(actual)):
        if not events[i]: continue
        for j in range(len(actual)):
            if i == j or actual[i] >= actual[j]: continue
            if predicted[i] < predicted[j]: con += 1
            elif predicted[i] > predicted[j]: dis += 1
            else: tie += 1
    total = con + dis + tie
    return (con + 0.5 * tie) / total if total > 0 else 0.5

def run_all_models(train, test_X, feats):
    preds = {}
    # 1. Cox PH
    try:
        m = CoxPHFitter(penalizer=0.1)
        m.fit(train[feats + ['tte_months', 'event']], 'tte_months', 'event')
        preds['Cox PH'] = float(_safe_pm(m, test_X)[0])
    except: preds['Cox PH'] = float(train['tte_months'].median())
    # 2. Weibull AFT
    try:
        m = WeibullAFTFitter(penalizer=0.05)
        m.fit(train[feats + ['tte_months', 'event']], 'tte_months', 'event')
        preds['Weibull AFT'] = float(_safe_pm(m, test_X)[0])
    except: preds['Weibull AFT'] = float(train['tte_months'].median())
    # 3. XGB Cox
    try:
        y = train['tte_months'].values.astype(float) * np.where(train['event'].values, 1, -1)
        dt = xgb.DMatrix(train[feats].values, label=y)
        ds = xgb.DMatrix(test_X.values)
        bst = xgb.train({'objective': 'survival:cox', 'tree_method': 'hist',
                          'max_depth': 2, 'learning_rate': 0.1,
                          'min_child_weight': 3, 'verbosity': 0}, dt, num_boost_round=20)
        hr = bst.predict(ds)[0]
        preds['XGB Cox'] = float(np.clip(train['tte_months'].median() / np.exp(hr), 1, CAP))
    except: preds['XGB Cox'] = float(train['tte_months'].median())
    # 4. XGB AFT
    try:
        yl = train['tte_months'].values.astype(float)
        yu = np.where(train['event'].values, yl, np.inf)
        dt = xgb.DMatrix(train[feats].values)
        dt.set_float_info('label_lower_bound', yl)
        dt.set_float_info('label_upper_bound', yu)
        ds = xgb.DMatrix(test_X.values)
        bst = xgb.train({'objective': 'survival:aft', 'aft_loss_distribution': 'normal',
                          'tree_method': 'hist', 'max_depth': 2,
                          'learning_rate': 0.1, 'verbosity': 0}, dt, num_boost_round=20)
        preds['XGB AFT'] = float(np.clip(bst.predict(ds)[0], 1, CAP))
    except: preds['XGB AFT'] = float(train['tte_months'].median())
    # 5. sksurv GBSA
    try:
        yt = Surv.from_arrays(train['event'].astype(bool), train['tte_months'].astype(float))
        m = GradientBoostingSurvivalAnalysis(n_estimators=50, max_depth=2, learning_rate=0.05,
                                             subsample=0.7, random_state=42)
        m.fit(train[feats].values, yt)
        risk = m.predict(test_X.values)[0]
        tr_r = m.predict(train[feats].values)
        std_r = max(np.std(tr_r), 1e-6)
        preds['sksurv GBSA'] = float(np.clip(
            train['tte_months'].median() * np.exp(-risk / std_r), 1, CAP))
    except: preds['sksurv GBSA'] = float(train['tte_months'].median())
    # 6. sksurv CWGB
    try:
        yt = Surv.from_arrays(train['event'].astype(bool), train['tte_months'].astype(float))
        m = ComponentwiseGradientBoostingSurvivalAnalysis(
            n_estimators=100, learning_rate=0.05, random_state=42)
        m.fit(train[feats].values, yt)
        risk = m.predict(test_X.values)[0]
        tr_r = m.predict(train[feats].values)
        std_r = max(np.std(tr_r), 1e-6)
        preds['sksurv CWGB'] = float(np.clip(
            train['tte_months'].median() * np.exp(-risk / std_r), 1, CAP))
    except: preds['sksurv CWGB'] = float(train['tte_months'].median())
    # 7. Stacked
    try:
        mc = CoxPHFitter(penalizer=0.1)
        mc.fit(train[feats + ['tte_months', 'event']], 'tte_months', 'event')
        cox_p = float(_safe_pm(mc, test_X)[0])
        te = train[train['event'] == 1]
        if len(te) < 5: preds['Stacked'] = cox_p
        else:
            cp = _safe_pm(mc, te[feats])
            res = np.log(te['tte_months'].values + 1) - np.log(cp + 1)
            tX = te[feats].copy()
            tX['hr'] = np.log(mc.predict_partial_hazard(te[feats]).values.flatten())
            sX = test_X.copy()
            sX['hr'] = np.log(float(mc.predict_partial_hazard(test_X).values[0]))
            dt = xgb.DMatrix(tX.values, label=res)
            ds = xgb.DMatrix(sX.values)
            bst = xgb.train({'max_depth': 1, 'learning_rate': 0.05, 'verbosity': 0,
                              'objective': 'reg:squarederror'}, dt, num_boost_round=10)
            rp = bst.predict(ds)[0]
            preds['Stacked'] = float(np.clip(cox_p * np.exp(rp), 1, CAP))
    except: preds['Stacked'] = float(train['tte_months'].median())
    return preds

# Run LOOCV
model_preds = {mn: {} for mn in MODEL_NAMES}  # {model: {well: pred}}
print('Running LOOCV across all 7 models...')
for i in range(len(cohort)):
    w = cohort.iloc[i]['well']
    train = cohort.drop(cohort.index[i]).reset_index(drop=True)
    test_X = cohort.iloc[[i]][FEATURES]
    fp = run_all_models(train, test_X, FEATURES)
    for mn in MODEL_NAMES:
        model_preds[mn][w] = fp[mn]
    if (i + 1) % 4 == 0:
        print(f'  Fold {i+1}/{len(cohort)} done')
print('All folds complete.')

# Reference data
wells = cohort.sort_values('tte_months')['well'].tolist()
actual = {w: float(cohort.loc[cohort['well'] == w, 'tte_months'].values[0]) for w in wells}
events = {w: int(cohort.loc[cohort['well'] == w, 'event'].values[0]) for w in wells}
evt_wells = [w for w in wells if events[w]]

# Show individual results
print(f'\n{"Model":18s} {"C-index":>8s} {"MAE":>8s} {"MedErr":>8s} {"MaxErr":>8s}')
print('-' * 55)
for mn in MODEL_NAMES:
    act_arr = np.array([actual[w] for w in wells])
    prd_arr = np.array([model_preds[mn][w] for w in wells])
    evt_arr = np.array([events[w] for w in wells])
    ci = harrell_cindex(act_arr, prd_arr, evt_arr)
    errs = [abs(model_preds[mn][w] - actual[w]) for w in evt_wells]
    print(f'{mn:18s} {ci:8.3f} {np.mean(errs):8.0f} {np.median(errs):8.0f} {np.max(errs):8.0f}')

Running LOOCV across all 7 models...


  Fold 4/16 done


  Fold 8/16 done


  Fold 12/16 done


  Fold 16/16 done
All folds complete.

Model               C-index      MAE   MedErr   MaxErr
-------------------------------------------------------
Cox PH                0.788      138       74      791
Weibull AFT           0.741      205       84      871
XGB Cox               0.703      153      157      310
XGB AFT               0.642      117      128      261
sksurv GBSA           0.844      179       75      791
sksurv CWGB           0.764      156       94      722
Stacked               0.774      104       72      410


---
## Section 4 — Combination Engine

Define ensemble strategies that work with any subset of models.
Then sweep all 120 subsets × 6 strategies = 720 experiments.

In [4]:
ENS_STRATEGIES = ['Simple Mean', 'Trimmed Mean', 'Median', 'Geometric Mean',
                  'Inv-MAE Weighted', 'Rank Fusion']

def compute_ensemble(subset_names, strategy, model_preds, wells, events, actual):
    """Compute ensemble predictions for a subset of models using a given strategy."""
    n_models = len(subset_names)
    preds = {}  # well -> predicted TTE

    # Gather predictions matrix
    pred_matrix = {}  # well -> array of preds from subset
    for w in wells:
        pred_matrix[w] = np.array([model_preds[mn][w] for mn in subset_names])

    # Compute inv-MAE weights
    evt_w = [w for w in wells if events[w]]
    model_maes = {}
    for mn in subset_names:
        errs = [abs(model_preds[mn][w] - actual[w]) for w in evt_w]
        model_maes[mn] = np.mean(errs) if errs else 1.0
    inv_w = np.array([1.0 / max(model_maes[mn], 1) for mn in subset_names])
    inv_w = inv_w / inv_w.sum()

    if strategy == 'Simple Mean':
        for w in wells:
            preds[w] = float(np.mean(pred_matrix[w]))

    elif strategy == 'Trimmed Mean':
        if n_models <= 2:
            # Can't trim with 2 models, fall back to mean
            for w in wells:
                preds[w] = float(np.mean(pred_matrix[w]))
        else:
            for w in wells:
                sv = np.sort(pred_matrix[w])
                preds[w] = float(np.mean(sv[1:-1]))

    elif strategy == 'Median':
        for w in wells:
            preds[w] = float(np.median(pred_matrix[w]))

    elif strategy == 'Geometric Mean':
        for w in wells:
            preds[w] = float(gmean(np.clip(pred_matrix[w], 1, CAP)))

    elif strategy == 'Inv-MAE Weighted':
        for w in wells:
            preds[w] = float(np.dot(inv_w, pred_matrix[w]))

    elif strategy == 'Rank Fusion':
        # Rank wells per model, average ranks, map back
        sorted_actuals = sorted(actual.values())
        model_ranks = {}
        for mn in subset_names:
            sw = sorted(wells, key=lambda w: model_preds[mn][w])
            model_ranks[mn] = {w: r + 1 for r, w in enumerate(sw)}
        avg_ranks = {w: np.mean([model_ranks[mn][w] for mn in subset_names])
                     for w in wells}
        rank_order = sorted(wells, key=lambda w: avg_ranks[w])
        n = len(rank_order)
        for idx, w in enumerate(rank_order):
            frac = idx / max(n - 1, 1)
            preds[w] = float(sorted_actuals[0] + frac * (sorted_actuals[-1] - sorted_actuals[0]))

    return preds

# Generate all subsets of size 2..7
all_subsets = []
for k in range(2, len(MODEL_NAMES) + 1):
    for combo in combinations(MODEL_NAMES, k):
        all_subsets.append(combo)

print(f"Model subsets by size:")
for k in range(2, 8):
    n = sum(1 for s in all_subsets if len(s) == k)
    print(f"  {k} models: {n} combinations")
print(f"  Total subsets: {len(all_subsets)}")
print(f"  × {len(ENS_STRATEGIES)} strategies = {len(all_subsets) * len(ENS_STRATEGIES)} experiments")

Model subsets by size:
  2 models: 21 combinations
  3 models: 35 combinations
  4 models: 35 combinations
  5 models: 21 combinations
  6 models: 7 combinations
  7 models: 1 combinations
  Total subsets: 120
  × 6 strategies = 720 experiments


---
## Section 5 — Full Combination Sweep

In [5]:
results = []
act_arr = np.array([actual[w] for w in wells])
evt_arr = np.array([events[w] for w in wells])

total = len(all_subsets) * len(ENS_STRATEGIES)
done = 0

for subset in all_subsets:
    subset_key = '+'.join([MODEL_SHORT[m] for m in subset])
    for strategy in ENS_STRATEGIES:
        preds = compute_ensemble(subset, strategy, model_preds, wells, events, actual)
        prd_arr = np.array([preds[w] for w in wells])
        ci = harrell_cindex(act_arr, prd_arr, evt_arr)
        errs = [abs(preds[w] - actual[w]) for w in evt_wells]
        mae = np.mean(errs)
        med_err = np.median(errs)
        max_err = np.max(errs)

        results.append({
            'subset': subset_key,
            'models': list(subset),
            'n_models': len(subset),
            'strategy': strategy,
            'c_index': round(ci, 3),
            'mae_events': round(mae, 1),
            'median_error': round(med_err, 1),
            'max_error': round(max_err, 1),
        })
        done += 1

    if done % 120 == 0:
        print(f"  {done}/{total} experiments done...")

rdf = pd.DataFrame(results)
print(f"\nCompleted {len(rdf)} experiments.")
print(f"C-index range: {rdf['c_index'].min():.3f} to {rdf['c_index'].max():.3f}")
print(f"MAE range: {rdf['mae_events'].min():.0f} to {rdf['mae_events'].max():.0f}")

  120/720 experiments done...
  240/720 experiments done...


  360/720 experiments done...


  480/720 experiments done...
  600/720 experiments done...


  720/720 experiments done...

Completed 720 experiments.
C-index range: 0.726 to 0.896
MAE range: 54 to 186


---
## Section 6 — Top Combinations

Best combinations by C-index and by MAE, plus the best at each subset size.

In [6]:
# ── Top 20 by C-index ──
print("=" * 95)
print("TOP 20 COMBINATIONS — by C-index")
print("=" * 95)
top_c = rdf.nlargest(20, 'c_index')
print(f"{'Rank':>4s} {'Subset':>35s} {'Strategy':>18s} {'N':>3s} "
      f"{'C-index':>8s} {'MAE':>7s} {'MedErr':>7s} {'MaxErr':>7s}")
print("-" * 95)
for rank, (_, row) in enumerate(top_c.iterrows(), 1):
    print(f"{rank:4d} {row['subset']:>35s} {row['strategy']:>18s} {row['n_models']:3d} "
          f"{row['c_index']:8.3f} {row['mae_events']:7.0f} {row['median_error']:7.0f} "
          f"{row['max_error']:7.0f}")

print()
print("=" * 95)
print("TOP 20 COMBINATIONS — by MAE")
print("=" * 95)
top_mae = rdf.nsmallest(20, 'mae_events')
print(f"{'Rank':>4s} {'Subset':>35s} {'Strategy':>18s} {'N':>3s} "
      f"{'C-index':>8s} {'MAE':>7s} {'MedErr':>7s} {'MaxErr':>7s}")
print("-" * 95)
for rank, (_, row) in enumerate(top_mae.iterrows(), 1):
    print(f"{rank:4d} {row['subset']:>35s} {row['strategy']:>18s} {row['n_models']:3d} "
          f"{row['c_index']:8.3f} {row['mae_events']:7.0f} {row['median_error']:7.0f} "
          f"{row['max_error']:7.0f}")

TOP 20 COMBINATIONS — by C-index
Rank                              Subset           Strategy   N  C-index     MAE  MedErr  MaxErr
-----------------------------------------------------------------------------------------------
   1                            Cox+GBSA        Rank Fusion   2    0.896      70      51     164
   2                            GBSA+Stk        Rank Fusion   2    0.877      79      69     164
   3                       Cox+XGBa+GBSA       Trimmed Mean   3    0.858     120      53     791
   4                       Cox+XGBa+GBSA             Median   3    0.858     120      53     791
   5                           XGBa+GBSA        Simple Mean   2    0.849      88      51     265
   6                           XGBa+GBSA       Trimmed Mean   2    0.849      88      51     265
   7                           XGBa+GBSA             Median   2    0.849      88      51     265
   8                      XGBa+GBSA+CWGB       Trimmed Mean   3    0.849     134      69     72

---
## Section 7 — Best Combination at Each Ensemble Size

In [7]:
print("=" * 100)
print("BEST COMBINATION BY ENSEMBLE SIZE")
print("=" * 100)

print(f"\n{'Size':>4s} | {'Best by C-index':^50s} | {'Best by MAE':^50s}")
print(f"{'':>4s} | {'Subset':>25s} {'Strat':>12s} {'C':>6s} {'MAE':>6s} | "
      f"{'Subset':>25s} {'Strat':>12s} {'C':>6s} {'MAE':>6s}")
print("-" * 100)

best_per_size = []
for k in range(2, 8):
    sub = rdf[rdf['n_models'] == k]
    bc = sub.loc[sub['c_index'].idxmax()]
    bm = sub.loc[sub['mae_events'].idxmin()]
    best_per_size.append({
        'size': k,
        'best_c_subset': bc['subset'], 'best_c_strategy': bc['strategy'],
        'best_c_cindex': bc['c_index'], 'best_c_mae': bc['mae_events'],
        'best_mae_subset': bm['subset'], 'best_mae_strategy': bm['strategy'],
        'best_mae_cindex': bm['c_index'], 'best_mae_mae': bm['mae_events'],
    })
    print(f"{k:4d} | {bc['subset']:>25s} {bc['strategy']:>12s} {bc['c_index']:6.3f} "
          f"{bc['mae_events']:6.0f} | {bm['subset']:>25s} {bm['strategy']:>12s} "
          f"{bm['c_index']:6.3f} {bm['mae_events']:6.0f}")

# Also show individual models for reference
print(f"\n{'1':>4s} | {'(individual models — no ensemble)':^50s}")
for mn in MODEL_NAMES:
    errs = [abs(model_preds[mn][w] - actual[w]) for w in evt_wells]
    ci = harrell_cindex(act_arr, np.array([model_preds[mn][w] for w in wells]), evt_arr)
    print(f"     | {MODEL_SHORT[mn]:>25s} {'(single)':>12s} {ci:6.3f} {np.mean(errs):6.0f}")

bps_df = pd.DataFrame(best_per_size)

BEST COMBINATION BY ENSEMBLE SIZE

Size |                  Best by C-index                   |                    Best by MAE                    
     |                    Subset        Strat      C    MAE |                    Subset        Strat      C    MAE
----------------------------------------------------------------------------------------------------
   2 |                  Cox+GBSA  Rank Fusion  0.896     70 |                 XGBa+CWGB Geometric Mean  0.802     54
   3 |             Cox+XGBa+GBSA Trimmed Mean  0.858    120 |             XGBa+GBSA+Stk Geometric Mean  0.821     64
   4 |        Cox+XGBc+XGBa+GBSA  Rank Fusion  0.840     77 |         Cox+XGBa+GBSA+Stk Geometric Mean  0.811     71
   5 |   Weib+XGBc+XGBa+GBSA+Stk Trimmed Mean  0.830     92 |    Cox+XGBa+GBSA+CWGB+Stk  Rank Fusion  0.821     77
   6 | Cox+Weib+XGBc+XGBa+GBSA+CWGB Inv-MAE Weighted  0.821    105 | Cox+Weib+XGBc+XGBa+GBSA+CWGB  Rank Fusion  0.792     78
   7 | Cox+Weib+XGBc+XGBa+GBSA+CWGB+Stk Trimmed

---
## Section 8 — Model Frequency: Which Models Appear in the Best Combinations?

Count how often each model appears in the top-N combinations.

In [8]:
# Top 50 by C-index
top50_c = rdf.nlargest(50, 'c_index')
# Top 50 by MAE
top50_mae = rdf.nsmallest(50, 'mae_events')

freq_c = {mn: 0 for mn in MODEL_NAMES}
freq_mae = {mn: 0 for mn in MODEL_NAMES}

for _, row in top50_c.iterrows():
    for mn in row['models']:
        freq_c[mn] += 1

for _, row in top50_mae.iterrows():
    for mn in row['models']:
        freq_mae[mn] += 1

print("MODEL FREQUENCY IN TOP-50 COMBINATIONS")
print("=" * 65)
print(f"{'Model':18s} {'Top-50 C-index':>15s} {'Top-50 MAE':>15s} {'Verdict':>15s}")
print("-" * 65)
for mn in MODEL_NAMES:
    fc = freq_c[mn]
    fm = freq_mae[mn]
    if fc >= 35 and fm >= 35:
        verdict = 'ESSENTIAL'
    elif fc >= 25 or fm >= 25:
        verdict = 'HELPFUL'
    elif fc >= 10 or fm >= 10:
        verdict = 'MARGINAL'
    else:
        verdict = 'DISPENSABLE'
    print(f"{mn:18s} {fc:>12d}/50 {fm:>12d}/50 {verdict:>15s}")

# Pairwise synergy: which pairs appear together most in top-50?
print("\n\nTOP MODEL PAIRS IN TOP-50 C-INDEX COMBINATIONS")
print("=" * 60)
pair_freq = {}
for _, row in top50_c.iterrows():
    for p in combinations(row['models'], 2):
        key = tuple(sorted(p))
        pair_freq[key] = pair_freq.get(key, 0) + 1

sorted_pairs = sorted(pair_freq.items(), key=lambda x: -x[1])[:15]
for (m1, m2), cnt in sorted_pairs:
    print(f"  {MODEL_SHORT[m1]:>5s} + {MODEL_SHORT[m2]:<5s} : {cnt}/50")

MODEL FREQUENCY IN TOP-50 COMBINATIONS
Model               Top-50 C-index      Top-50 MAE         Verdict
-----------------------------------------------------------------
Cox PH                       12/50           22/50        MARGINAL
Weibull AFT                   9/50           14/50        MARGINAL
XGB Cox                      17/50           13/50        MARGINAL
XGB AFT                      41/50           47/50       ESSENTIAL
sksurv GBSA                  37/50           28/50         HELPFUL
sksurv CWGB                  10/50           26/50         HELPFUL
Stacked                      23/50           19/50        MARGINAL


TOP MODEL PAIRS IN TOP-50 C-INDEX COMBINATIONS
   XGBa + GBSA  : 28/50
   XGBa + XGBc  : 17/50
    Stk + XGBa  : 17/50
    Stk + GBSA  : 16/50
    Cox + XGBa  : 10/50
   XGBa + CWGB  : 10/50
   XGBc + GBSA  : 10/50
    Stk + XGBc  : 10/50
    Cox + GBSA  : 9/50
   Weib + XGBa  : 8/50
   Weib + XGBc  : 6/50
   CWGB + GBSA  : 5/50
   Weib + GBSA  : 5/50
   

---
## Section 9 — Which Ensemble Strategy Works Best at Each Size?

In [9]:
print("BEST STRATEGY BY ENSEMBLE SIZE")
print("=" * 80)

# For each size, which strategy has the best average C-index and best average MAE?
print(f"\n{'Size':>4s} | {'Best Strategy (C-index)':>25s} {'Avg C':>7s} | "
      f"{'Best Strategy (MAE)':>25s} {'Avg MAE':>8s}")
print("-" * 80)

for k in range(2, 8):
    sub = rdf[rdf['n_models'] == k]
    strat_c = sub.groupby('strategy')['c_index'].mean().sort_values(ascending=False)
    strat_mae = sub.groupby('strategy')['mae_events'].mean().sort_values()
    print(f"{k:4d} | {strat_c.index[0]:>25s} {strat_c.iloc[0]:7.3f} | "
          f"{strat_mae.index[0]:>25s} {strat_mae.iloc[0]:8.0f}")

# Overall strategy ranking
print("\n\nOVERALL STRATEGY RANKING (averaged across all subset sizes)")
print("=" * 60)
strat_overall_c = rdf.groupby('strategy')['c_index'].mean().sort_values(ascending=False)
strat_overall_mae = rdf.groupby('strategy')['mae_events'].mean().sort_values()

print(f"\n{'By C-index':>30s}  |  {'By MAE':>30s}")
print("-" * 65)
for i in range(len(ENS_STRATEGIES)):
    sc = strat_overall_c.index[i]
    sm = strat_overall_mae.index[i]
    print(f"{sc:>25s} {strat_overall_c.iloc[i]:.3f}  |  {sm:>25s} {strat_overall_mae.iloc[i]:.0f}")

BEST STRATEGY BY ENSEMBLE SIZE

Size |   Best Strategy (C-index)   Avg C |       Best Strategy (MAE)  Avg MAE
--------------------------------------------------------------------------------
   2 |               Rank Fusion   0.800 |               Rank Fusion       91
   3 |               Rank Fusion   0.796 |               Rank Fusion       87
   4 |                    Median   0.801 |               Rank Fusion       85
   5 |          Inv-MAE Weighted   0.803 |               Rank Fusion       84
   6 |              Trimmed Mean   0.806 |               Rank Fusion       83
   7 |          Inv-MAE Weighted   0.802 |               Rank Fusion       82


OVERALL STRATEGY RANKING (averaged across all subset sizes)

                    By C-index  |                          By MAE
-----------------------------------------------------------------
              Rank Fusion 0.798  |                Rank Fusion 87
         Inv-MAE Weighted 0.797  |             Geometric Mean 100
             Tr

---
## Section 10 — Per-Well Predictions: Best Combinations vs Baseline

In [10]:
# Get best by C and best by MAE
best_c_row = rdf.loc[rdf['c_index'].idxmax()]
best_mae_row = rdf.loc[rdf['mae_events'].idxmin()]
# Also get best 3-model combo (simpler)
best_3_c = rdf[rdf['n_models'] == 3].loc[rdf[rdf['n_models'] == 3]['c_index'].idxmax()]
best_3_mae = rdf[rdf['n_models'] == 3].loc[rdf[rdf['n_models'] == 3]['mae_events'].idxmin()]

combos_to_show = [
    ('Best C-index overall', best_c_row),
    ('Best MAE overall', best_mae_row),
    ('Best 3-model (C)', best_3_c),
    ('Best 3-model (MAE)', best_3_mae),
]

for label, row in combos_to_show:
    subset = row['models']
    strategy = row['strategy']
    preds = compute_ensemble(subset, strategy, model_preds, wells, events, actual)
    ci = row['c_index']
    mae = row['mae_events']

    print(f"\n{'='*80}")
    print(f"{label}: {row['subset']} — {strategy}")
    print(f"C-index={ci:.3f}, MAE={mae:.0f}, N_models={row['n_models']}")
    print(f"{'='*80}")
    print(f"{'Well':12s} {'Evt':>4s} {'Actual':>7s} {'Pred':>7s} {'Error':>8s}")
    print("-" * 42)
    for w in wells:
        a = actual[w]
        p = preds[w]
        e = 'Y' if events[w] else 'N'
        print(f"{w:12s} {e:>4s} {a:7.0f} {p:7.0f} {p-a:+8.0f}")


Best C-index overall: Cox+GBSA — Rank Fusion
C-index=0.896, MAE=70, N_models=2
Well          Evt  Actual    Pred    Error
------------------------------------------
M-67-HRL        Y      45      45       +0
M-82-HRL        Y      48      79      +31
M-81-HRL        Y      57     114      +57
M-58-HRL        Y      88     252     +164
M-75-HRL        Y     127     148      +21
M-E-2-HRL       N     147     183      +36
M-65-HRL        Y     259     286      +27
M-63-HRL        Y     275     321      +46
M-61-HRL        Y     284     424     +140
M-56-HRL        Y     299     217      -82
M-41-HRL        Y     329     459     +130
M-50-HRL        Y     379     355      -24
M-57-HRL        N     383     390       +7
M-11-HRL        Y     409     528     +119
M-22-HRL        N     524     493      -31
M-13-HRL        N     562     562       +0

Best MAE overall: XGBa+CWGB — Geometric Mean
C-index=0.802, MAE=54, N_models=2
Well          Evt  Actual    Pred    Error
-----------------------

---
## Section 11 — Visualizations

In [11]:
# ── Figure 1: C-index and MAE vs ensemble size ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplots of C-index by size
sizes = sorted(rdf['n_models'].unique())
data_c = [rdf[rdf['n_models'] == k]['c_index'].values for k in sizes]
data_m = [rdf[rdf['n_models'] == k]['mae_events'].values for k in sizes]

bp1 = axes[0].boxplot(data_c, positions=sizes, widths=0.5, patch_artist=True)
for box in bp1['boxes']:
    box.set_facecolor('lightblue')
# Add individual model reference lines
for mn in MODEL_NAMES:
    ci = harrell_cindex(act_arr, np.array([model_preds[mn][w] for w in wells]), evt_arr)
    axes[0].axhline(y=ci, color='gray', linestyle=':', alpha=0.3)
axes[0].set_xlabel('Number of Models in Ensemble')
axes[0].set_ylabel('C-index')
axes[0].set_title('C-index Distribution by Ensemble Size')
axes[0].axhline(y=0.788, color='red', linestyle='--', alpha=0.7, label='Cox PH baseline')
axes[0].legend(fontsize=8)

bp2 = axes[1].boxplot(data_m, positions=sizes, widths=0.5, patch_artist=True)
for box in bp2['boxes']:
    box.set_facecolor('lightyellow')
axes[1].set_xlabel('Number of Models in Ensemble')
axes[1].set_ylabel('MAE (months, events only)')
axes[1].set_title('MAE Distribution by Ensemble Size')
axes[1].axhline(y=138, color='red', linestyle='--', alpha=0.7, label='Cox PH baseline')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / '01_performance_by_size.png')
print(f"Saved {FIG_DIR / '01_performance_by_size.png'}")
plt.close()

# ── Figure 2: C-index vs MAE scatter colored by size ──
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, 6))
for ki, k in enumerate(sizes):
    sub = rdf[rdf['n_models'] == k]
    ax.scatter(sub['mae_events'], sub['c_index'], c=[colors[ki]], s=30,
               alpha=0.5, label=f'{k} models', edgecolors='k', linewidth=0.2)
# Mark the best
bc = rdf.loc[rdf['c_index'].idxmax()]
bm = rdf.loc[rdf['mae_events'].idxmin()]
ax.scatter(bc['mae_events'], bc['c_index'], c='red', marker='*', s=250,
           zorder=10, label=f"Best C: {bc['subset']}")
ax.scatter(bm['mae_events'], bm['c_index'], c='blue', marker='*', s=250,
           zorder=10, label=f"Best MAE: {bm['subset']}")
# Baseline
ax.scatter(138, 0.788, c='black', marker='D', s=100, zorder=10, label='Cox PH alone')
ax.set_xlabel('MAE (months)')
ax.set_ylabel('C-index')
ax.set_title('All 720 Ensemble Combinations — C-index vs MAE')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / '02_cindex_vs_mae_all.png')
print(f"Saved {FIG_DIR / '02_cindex_vs_mae_all.png'}")
plt.close()

# ── Figure 3: Model frequency in top-50 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_sorted_c = sorted(freq_c.items(), key=lambda x: -x[1])
models_sorted_mae = sorted(freq_mae.items(), key=lambda x: -x[1])

axes[0].barh([MODEL_SHORT[m] for m, _ in models_sorted_c],
             [c for _, c in models_sorted_c], color='steelblue')
axes[0].set_xlabel('Frequency in Top-50 (C-index)')
axes[0].set_title('Model Frequency — Top 50 by C-index')
axes[0].axvline(x=25, color='gray', linestyle='--', alpha=0.5)

axes[1].barh([MODEL_SHORT[m] for m, _ in models_sorted_mae],
             [c for _, c in models_sorted_mae], color='coral')
axes[1].set_xlabel('Frequency in Top-50 (MAE)')
axes[1].set_title('Model Frequency — Top 50 by MAE')
axes[1].axvline(x=25, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(FIG_DIR / '03_model_frequency.png')
print(f"Saved {FIG_DIR / '03_model_frequency.png'}")
plt.close()

# ── Figure 4: Best at each size ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sz = [r['size'] for r in best_per_size]
bc_vals = [r['best_c_cindex'] for r in best_per_size]
bm_vals = [r['best_mae_mae'] for r in best_per_size]
bc_mae = [r['best_c_mae'] for r in best_per_size]
bm_c = [r['best_mae_cindex'] for r in best_per_size]

axes[0].plot(sz, bc_vals, 'o-', color='steelblue', label='Best C-index', linewidth=2)
axes[0].plot(sz, bm_c, 's--', color='coral', label='C-index of MAE-best', linewidth=1.5)
axes[0].axhline(y=0.788, color='gray', linestyle=':', label='Cox PH baseline')
axes[0].set_xlabel('Ensemble Size')
axes[0].set_ylabel('C-index')
axes[0].set_title('Best C-index at Each Ensemble Size')
axes[0].legend(fontsize=8)
axes[0].set_xticks(sz)

axes[1].plot(sz, bm_vals, 's-', color='coral', label='Best MAE', linewidth=2)
axes[1].plot(sz, bc_mae, 'o--', color='steelblue', label='MAE of C-best', linewidth=1.5)
axes[1].axhline(y=138, color='gray', linestyle=':', label='Cox PH baseline')
axes[1].set_xlabel('Ensemble Size')
axes[1].set_ylabel('MAE (months)')
axes[1].set_title('Best MAE at Each Ensemble Size')
axes[1].legend(fontsize=8)
axes[1].set_xticks(sz)

plt.tight_layout()
plt.savefig(FIG_DIR / '04_best_by_size.png')
print(f"Saved {FIG_DIR / '04_best_by_size.png'}")
plt.close()

Saved ensemble_analysis/figures/01_performance_by_size.png


Saved ensemble_analysis/figures/02_cindex_vs_mae_all.png
Saved ensemble_analysis/figures/03_model_frequency.png


Saved ensemble_analysis/figures/04_best_by_size.png


---
## Section 12 — Save All Results

In [12]:
outpath = RES_DIR / 'combination_sweep_results.xlsx'
with pd.ExcelWriter(outpath, engine='openpyxl') as writer:
    # Sheet 1: All 720 experiments
    rdf_out = rdf.drop(columns=['models'])
    rdf_out.to_excel(writer, sheet_name='All_720', index=False)

    # Sheet 2: Top 20 by C-index
    top_c.drop(columns=['models']).to_excel(writer, sheet_name='Top20_Cindex', index=False)

    # Sheet 3: Top 20 by MAE
    top_mae.drop(columns=['models']).to_excel(writer, sheet_name='Top20_MAE', index=False)

    # Sheet 4: Best per size
    bps_df.to_excel(writer, sheet_name='Best_Per_Size', index=False)

    # Sheet 5: Per-well predictions for the overall best (C-index)
    best_combo = rdf.loc[rdf['c_index'].idxmax()]
    preds_best_c = compute_ensemble(best_combo['models'], best_combo['strategy'],
                                     model_preds, wells, events, actual)
    best_mae_combo = rdf.loc[rdf['mae_events'].idxmin()]
    preds_best_mae = compute_ensemble(best_mae_combo['models'], best_mae_combo['strategy'],
                                       model_preds, wells, events, actual)

    pw_rows = []
    for w in wells:
        pw_rows.append({
            'Well': w,
            'Event': 'Y' if events[w] else 'N',
            'Actual': actual[w],
            f"Best_C ({best_combo['subset']}, {best_combo['strategy']})": round(preds_best_c[w]),
            f"Best_MAE ({best_mae_combo['subset']}, {best_mae_combo['strategy']})": round(preds_best_mae[w]),
            'Cox PH': round(model_preds['Cox PH'][w]),
            '7-model Rank Fusion': round(compute_ensemble(
                tuple(MODEL_NAMES), 'Rank Fusion', model_preds, wells, events, actual)[w]),
        })
    pd.DataFrame(pw_rows).to_excel(writer, sheet_name='Per_Well_Best', index=False)

    # Sheet 6: Strategy comparison
    strat_stats = rdf.groupby('strategy').agg(
        mean_c=('c_index', 'mean'), max_c=('c_index', 'max'),
        mean_mae=('mae_events', 'mean'), min_mae=('mae_events', 'min')
    ).sort_values('mean_c', ascending=False)
    strat_stats.to_excel(writer, sheet_name='Strategy_Comparison')

print(f"Saved: {outpath}")

# CSV backup
rdf.drop(columns=['models']).to_csv(RES_DIR / 'combination_sweep_all.csv', index=False)
print(f"Saved: {RES_DIR / 'combination_sweep_all.csv'}")

Saved: ensemble_analysis/results/combination_sweep_results.xlsx
Saved: ensemble_analysis/results/combination_sweep_all.csv


---
## Section 13 — Summary and Recommendations

In [13]:
print("=" * 80)
print("MODEL COMBINATION SWEEP — FINAL SUMMARY")
print("=" * 80)

best_c = rdf.loc[rdf['c_index'].idxmax()]
best_m = rdf.loc[rdf['mae_events'].idxmin()]

print(f"""
OVERALL BEST C-INDEX:
  Combination: {best_c['subset']}
  Strategy:    {best_c['strategy']}
  N models:    {best_c['n_models']}
  C-index:     {best_c['c_index']:.3f}
  MAE:         {best_c['mae_events']:.0f} months

OVERALL BEST MAE:
  Combination: {best_m['subset']}
  Strategy:    {best_m['strategy']}
  N models:    {best_m['n_models']}
  C-index:     {best_m['c_index']:.3f}
  MAE:         {best_m['mae_events']:.0f} months

BASELINE COMPARISON:
  Cox PH alone:        C=0.788, MAE=138
  7-model Rank Fusion: C=0.783, MAE=59 (from ensemble notebook)
""")

# Pareto-optimal combinations (not dominated on both C and MAE)
print("PARETO-OPTIMAL COMBINATIONS (best tradeoff between C-index and MAE):")
print("-" * 80)
pareto = []
for _, row in rdf.iterrows():
    dominated = False
    for _, other in rdf.iterrows():
        if other['c_index'] > row['c_index'] and other['mae_events'] < row['mae_events']:
            dominated = True
            break
    if not dominated:
        pareto.append(row)
pareto_df = pd.DataFrame(pareto).sort_values('c_index', ascending=False)
print(f"{'Subset':>35s} {'Strategy':>18s} {'N':>3s} {'C-index':>8s} {'MAE':>7s}")
for _, row in pareto_df.head(10).iterrows():
    print(f"{row['subset']:>35s} {row['strategy']:>18s} {row['n_models']:3.0f} "
          f"{row['c_index']:8.3f} {row['mae_events']:7.0f}")

print("\n\nKEY FINDINGS:")
print("1. Fewer models can outperform the full 7-model ensemble")
print("2. The optimal ensemble size depends on the metric (C-index vs MAE)")
print("3. Some models consistently appear in top combinations (essential)")
print("4. Some models rarely appear (can be safely dropped)")
print("5. Strategy choice matters — Rank Fusion and Geometric Mean tend to be robust")

MODEL COMBINATION SWEEP — FINAL SUMMARY

OVERALL BEST C-INDEX:
  Combination: Cox+GBSA
  Strategy:    Rank Fusion
  N models:    2
  C-index:     0.896
  MAE:         70 months

OVERALL BEST MAE:
  Combination: XGBa+CWGB
  Strategy:    Geometric Mean
  N models:    2
  C-index:     0.802
  MAE:         54 months

BASELINE COMPARISON:
  Cox PH alone:        C=0.788, MAE=138
  7-model Rank Fusion: C=0.783, MAE=59 (from ensemble notebook)

PARETO-OPTIMAL COMBINATIONS (best tradeoff between C-index and MAE):
--------------------------------------------------------------------------------


                             Subset           Strategy   N  C-index     MAE
                           Cox+GBSA        Rank Fusion   2    0.896      70
                      Cox+XGBa+GBSA     Geometric Mean   3    0.821      65
                      XGBa+GBSA+Stk     Geometric Mean   3    0.821      64
                          XGBa+GBSA     Geometric Mean   2    0.811      59
                          XGBa+CWGB     Geometric Mean   2    0.802      54


KEY FINDINGS:
1. Fewer models can outperform the full 7-model ensemble
2. The optimal ensemble size depends on the metric (C-index vs MAE)
3. Some models consistently appear in top combinations (essential)
4. Some models rarely appear (can be safely dropped)
5. Strategy choice matters — Rank Fusion and Geometric Mean tend to be robust
